# Aspect-Based Sentiment Analysis on Amazon Reviews

Using Llama 3 8B to generate silver labels, then fine-tuning ModernBERT on them.

**Pipeline**

1. Hold out a manual verification set of 100 reviews, before any labeling
2. Sample 2,000 training reviews per category, stratified by sub-category
3. Generate aspect-sentiment labels with Llama 3 8B running locally via Ollama
4. Clean invalid labels and format as `review [SEP] aspect`
5. Split 80/10/10 stratified by sentiment
6. Fine-tune ModernBERT with a 3-class head
7. Evaluate per category, against 4 traditional baselines, and against direct LLM prompting
8. Label agreement, confusion matrices, trend analysis, aspect rankings

**Requirements**

```
pip install pandas transformers datasets torch scikit-learn matplotlib requests
ollama pull llama3.1:8b
```

Ollama must be running on `localhost:11434`. Set `base_path` below to the folder
holding the Amazon Reviews 2023 JSONL files.

**Runtime** — silver labeling is the slow stage at roughly 95 minutes for 10,000
reviews with 8 parallel workers. Fine-tuning takes about 10 minutes on an
RTX 4080 Super.

In [ ]:
import pandas as pd
import json
import requests
import concurrent.futures
import time

base_path = r"C:\Users\sapds\Capstone"

category_files = {
    "Electronics": ("Electronics.jsonl", "meta_Electronics.jsonl"),
    "Beauty_and_Personal_Care": ("Beauty_and_Personal_Care.jsonl", "meta_Beauty_and_Personal_Care.jsonl"),
    "Video_Games": ("Video_Games.jsonl", "meta_Video_Games.jsonl"),
    "Amazon_Fashion": ("Amazon_Fashion.jsonl", "meta_Amazon_Fashion.jsonl"),
    "Books": ("Books.jsonl", "meta_Books.jsonl")
}

categories = list(category_files.keys())

timing_log = {}

## Helper functions

`load_jsonl` reads the first 50,000 rows of a JSONL file.
`get_subcategory` pulls the sub-category out of the nested metadata list.

In [ ]:
def load_jsonl(filepath, max_rows=50000):
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_rows:
                break
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


def get_subcategory(category_list):
    try:
        return category_list[0][1]
    except:
        return "Unknown"

## Step 1: Hold out the manual validation set

This runs first so these reviews never appear in labeling or training.
50 per category, sampled across sub-categories.

In [ ]:
start = time.time()

validation_samples = []
held_out_asins = {}

for category_name, (review_file, meta_file) in category_files.items():
    print(f"Processing {category_name}")

    reviews = load_jsonl(f"{base_path}\\{review_file}")
    reviews['word_count'] = reviews['text'].str.split().str.len()
    reviews = reviews[reviews['word_count'] >= 10]

    meta = load_jsonl(f"{base_path}\\{meta_file}")
    reviews = reviews.merge(meta[['parent_asin', 'categories']], on='parent_asin', how='left')
    reviews['subcategory'] = reviews['categories'].apply(get_subcategory)

    sample = reviews.groupby('subcategory', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 10), random_state=99)
    ).head(50)

    sample['category'] = category_name
    validation_samples.append(sample)
    held_out_asins[category_name] = sample['asin'].tolist()

validation_df = pd.concat(validation_samples, ignore_index=True)
validation_df['my_aspects'] = ""
validation_df['my_sentiments'] = ""

validation_df[['asin', 'text', 'rating', 'subcategory', 'category', 'my_aspects', 'my_sentiments']].to_csv(
    f"{base_path}\\manual_validation_set.csv", index=False
)

timing_log['validation_set_creation'] = time.time() - start
print(f"\nValidation set saved: {len(validation_df)} reviews")
print(f"Time: {timing_log['validation_set_creation']:.1f}s")

## Step 2: Sample training data

2,000 reviews per category, stratified by sub-category so one popular
sub-category doesn't dominate. Validation ASINs are excluded here.

In [ ]:
start = time.time()

for category_name, (review_file, meta_file) in category_files.items():
    print(f"Processing {category_name}")

    reviews = load_jsonl(f"{base_path}\\{review_file}")
    reviews['word_count'] = reviews['text'].str.split().str.len()
    reviews = reviews[reviews['word_count'] >= 10]

    meta = load_jsonl(f"{base_path}\\{meta_file}")
    reviews = reviews.merge(meta[['parent_asin', 'categories']], on='parent_asin', how='left')
    reviews['subcategory'] = reviews['categories'].apply(get_subcategory)

    # exclude validation set reviews
    reviews = reviews[~reviews['asin'].isin(held_out_asins[category_name])]

    sample = reviews.groupby('subcategory', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 200), random_state=42)
    )

    if len(sample) < 2000:
        extra = reviews[~reviews.index.isin(sample.index)].sample(2000 - len(sample), random_state=42)
        sample = pd.concat([sample, extra])
    sample = sample.head(2000)

    cols = ['asin', 'parent_asin', 'text', 'rating', 'timestamp', 'title', 'subcategory']
    sample[cols].to_csv(f"{base_path}\\{category_name}_sampled.csv", index=False)
    print(f"  Saved {len(sample)} reviews")

timing_log['training_sampling'] = time.time() - start
print(f"\nAll categories sampled in {timing_log['training_sampling']:.1f}s")

## Step 3: Silver label generation

Llama 3 8B runs locally through Ollama. For each review it returns every
aspect mentioned plus a sentiment for each, as JSON. Temperature is 0 so
the same review always gives the same labels.

In [ ]:
def get_aspects(review_text):
    prompt = f"""Analyze this Amazon review and extract every aspect (product feature) mentioned.
For each aspect, classify the sentiment as positive, neutral, or negative.

Sentiment definitions:
- positive: the reviewer clearly likes or praises the aspect
- neutral: the reviewer has mixed feelings, says it is okay/average/acceptable, or mentions it without strong opinion
- negative: the reviewer clearly dislikes or complains about the aspect

Example:
Review: "The camera is great but the battery is just okay and the screen is terrible"
Output: {{"aspects": [{{"aspect": "camera", "sentiment": "positive"}}, {{"aspect": "battery", "sentiment": "neutral"}}, {{"aspect": "screen", "sentiment": "negative"}}]}}

Return ONLY valid JSON in this exact format, no other text:
{{"aspects": [{{"aspect": "battery life", "sentiment": "positive"}}, {{"aspect": "sound quality", "sentiment": "negative"}}]}}

If no aspects are found, return: {{"aspects": []}}

Review: {review_text}"""

    response = requests.post("http://localhost:11434/api/generate", json={
        "model": "llama3.1:8b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0}
    })

    text = response.json()["response"]
    result = json.loads(text)
    return result["aspects"]

In [ ]:
test_review = "The battery life is amazing but the screen is too dim and the speakers are just okay"
print(get_aspects(test_review))

In [ ]:
def process_review(row, category_name):
    try:
        aspects = get_aspects(row['text'])
    except:
        return None

    rows = []
    for a in aspects:
        if 'aspect' not in a or 'sentiment' not in a:
            continue
        rows.append({
            'review_text': row['text'],
            'aspect': a['aspect'],
            'sentiment': a['sentiment'],
            'rating': row['rating'],
            'timestamp': row['timestamp'],
            'asin': row['asin'],
            'category': category_name
        })
    return rows

### Run labeling on all 5 categories

8 parallel workers. Takes roughly 95 minutes total on the 4080.

In [ ]:
silver_label_timing = {}

for category_name in categories:
    print(f"\nProcessing {category_name}")
    start = time.time()

    df = pd.read_csv(f"{base_path}\\{category_name}_sampled.csv")

    results = []
    failed = 0
    completed = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        futures = []
        for _, row in df.iterrows():
            futures.append(executor.submit(process_review, row, category_name))

        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            if result is None:
                failed += 1
            else:
                results.extend(result)

            completed += 1
            if completed % 200 == 0:
                print(f"  Progress: {completed}/{len(df)}")

    elapsed = time.time() - start
    silver_label_timing[category_name] = elapsed

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{base_path}\\{category_name}_silver_labels.csv", index=False)
    print(f"  Done: {len(results_df)} pairs, {failed} failed, {elapsed:.1f}s")

timing_log['silver_labels_per_category'] = silver_label_timing
timing_log['silver_labels_total'] = sum(silver_label_timing.values())

print(f"\nTotal silver labeling time: {timing_log['silver_labels_total']:.1f}s")
print("Per category:")
for category_name, t in silver_label_timing.items():
    print(f"  {category_name}: {t:.1f}s")

## Step 4: Combine and clean the labels

Merge all 5 category files, then drop any row where Llama returned a
sentiment outside positive / neutral / negative.

In [ ]:
all_data = []
for category_name in categories:
    df = pd.read_csv(f"{base_path}\\{category_name}_silver_labels.csv")
    all_data.append(df)

silver_df = pd.concat(all_data, ignore_index=True)
print(f"Total: {len(silver_df)} pairs")
print(silver_df['sentiment'].value_counts())

In [ ]:
valid_sentiments = ['positive', 'neutral', 'negative']
silver_df = silver_df[silver_df['sentiment'].isin(valid_sentiments)]
print(f"After cleaning: {len(silver_df)} pairs")

### Format for ModernBERT

Each row becomes `review [SEP] aspect` with the sentiment as the label.

In [ ]:
silver_df['input_text'] = silver_df['review_text'] + " [SEP] " + silver_df['aspect']
training_df = silver_df[['input_text', 'sentiment', 'category', 'aspect', 'review_text']].copy()
print(f"Example: {training_df.iloc[0]['input_text'][:150]}")
print(f"Label: {training_df.iloc[0]['sentiment']}")

## Step 5: Train / validation / test split

80/10/10, stratified by sentiment so all three splits have the same class balance.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    training_df, test_size=0.2, random_state=0, stratify=training_df['sentiment']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=0, stratify=temp_df['sentiment']
)

train_df.to_csv(f"{base_path}\\train.csv", index=False)
val_df.to_csv(f"{base_path}\\val.csv", index=False)
test_df.to_csv(f"{base_path}\\test.csv", index=False)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## Step 6: Fine-tune ModernBERT

ModernBERT-base with a 3-class head. 3 epochs, batch size 16, learning rate 2e-5,
fp16 mixed precision.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

model_name = "answerdotai/ModernBERT-base"
label_to_id = {"negative": 0, "neutral": 1, "positive": 2}
id_to_label = {0: "negative", 1: "neutral", 2: "positive"}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id_to_label, label2id=label_to_id
)
print("Model loaded")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
def make_dataset(df):
    ds = Dataset.from_pandas(df[['input_text', 'sentiment']].reset_index(drop=True))
    ds = ds.map(lambda x: {'labels': label_to_id[x['sentiment']]})
    return ds

train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

def tokenize(batch):
    return tokenizer(batch['input_text'], padding='max_length', truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    return {'accuracy': accuracy, 'f1_macro': f1, 'precision_macro': precision, 'recall_macro': recall}

In [ ]:
training_args = TrainingArguments(
    output_dir=f"{base_path}\\modernbert_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

start = time.time()
trainer.train()
timing_log['modernbert_training'] = time.time() - start
print(f"\nModernBERT training time: {timing_log['modernbert_training']:.1f}s")

In [ ]:
results = trainer.evaluate(test_ds)

trainer.save_model(f"{base_path}\\modernbert_finetuned")
tokenizer.save_pretrained(f"{base_path}\\modernbert_finetuned")
print("Model saved")

## Step 7: Per-category evaluation

Splits the test set by category to check the model isn't only good at one domain.

In [ ]:
modernbert_per_category = []

for category_name in categories:
    cat_test = test_df[test_df['category'] == category_name]

    ds = Dataset.from_pandas(cat_test[['input_text', 'sentiment']].reset_index(drop=True))
    ds = ds.map(lambda x: {'labels': label_to_id[x['sentiment']]})
    ds = ds.map(lambda b: tokenizer(b['input_text'], padding='max_length', truncation=True, max_length=256), batched=True)

    start = time.time()
    preds = trainer.predict(ds)
    inference_time = time.time() - start

    pred_labels = np.argmax(preds.predictions, axis=1)
    acc = accuracy_score(preds.label_ids, pred_labels)
    prec, rec, f1, _ = precision_recall_fscore_support(preds.label_ids, pred_labels, average='macro', zero_division=0)

    modernbert_per_category.append({
        'category': category_name,
        'samples': len(cat_test),
        'accuracy': acc,
        'f1': f1,
        'precision': prec,
        'recall': rec,
        'inference_seconds': inference_time,
        'reviews_per_second': len(cat_test) / inference_time
    })

modernbert_df = pd.DataFrame(modernbert_per_category)
modernbert_df.to_csv(f"{base_path}\\modernbert_per_category.csv", index=False)
modernbert_df

### Overall ROC-AUC for ModernBERT

Softmax over the logits to get probabilities, which ROC-AUC needs.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
from sklearn.metrics import roc_auc_score
import torch
import numpy as np

model_path = f"{base_path}\\modernbert_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
trainer = Trainer(model=model)

# get predictions with probabilities
test_ds = Dataset.from_pandas(test_df[['input_text', 'sentiment']].reset_index(drop=True))
test_ds = test_ds.map(lambda x: {'labels': label_to_id[x['sentiment']]})
test_ds = test_ds.map(lambda b: tokenizer(b['input_text'], padding='max_length', truncation=True, max_length=256), batched=True)

preds = trainer.predict(test_ds)

# softmax to get probabilities
probabilities = torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()
true_labels = preds.label_ids

auc = roc_auc_score(true_labels, probabilities, multi_class='ovr', average='macro')
print(f"ModernBERT ROC-AUC: {auc:.4f}")

## Step 8: Baseline models

Logistic Regression, SVM, Random Forest, and a stacking ensemble.
TF-IDF features, trained separately per category.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

baseline_per_category = []
baseline_start = time.time()

for category_name in categories:
    category_train = train_df[train_df['category'] == category_name]
    category_test = test_df[test_df['category'] == category_name]

    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(category_train['input_text'])
    X_test = vectorizer.transform(category_test['input_text'])
    y_train = category_train['sentiment'].map(label_to_id)
    y_test = category_test['sentiment'].map(label_to_id)

    logistic = LogisticRegression(max_iter=1000)
    svm = CalibratedClassifierCV(LinearSVC(max_iter=2000))
    random_forest = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0)
    stacking = StackingClassifier(
        estimators=[('lr', logistic), ('svm', svm), ('rf', random_forest)],
        final_estimator=LogisticRegression(max_iter=1000)
    )

    models = [
        ("Logistic Regression", logistic),
        ("SVM", svm),
        ("Random Forest", random_forest),
        ("Stacking", stacking)
    ]

    for model_name, m in models:
        train_start = time.time()
        m.fit(X_train, y_train)
        train_time = time.time() - train_start

        inf_start = time.time()
        predictions = m.predict(X_test)
        probabilities = m.predict_proba(X_test)
        inf_time = time.time() - inf_start

        acc = accuracy_score(y_test, predictions)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, predictions, average='macro', zero_division=0)
        try:
            auc = roc_auc_score(y_test, probabilities, multi_class='ovr', average='macro')
        except:
            auc = None

        baseline_per_category.append({
            'category': category_name,
            'model': model_name,
            'accuracy': acc,
            'f1': f1,
            'precision': prec,
            'recall': rec,
            'roc_auc': auc,
            'training_seconds': train_time,
            'inference_seconds': inf_time
        })

timing_log['baselines_total'] = time.time() - baseline_start

baseline_df_results = pd.DataFrame(baseline_per_category)
baseline_df_results.to_csv(f"{base_path}\\baseline_per_category.csv", index=False)
print(f"\nTotal baseline time: {timing_log['baselines_total']:.1f}s")
baseline_df_results

In [ ]:
baseline_pivot = baseline_df_results.pivot(index='category', columns='model', values='accuracy')
print("Accuracy by category and model:")
baseline_pivot

## Step 9: Direct Llama baseline

No fine-tuning, just prompting Llama for each test example. 500 samples
since it's much slower than the trained models.

In [ ]:
import concurrent.futures

llama_sample = test_df.sample(500, random_state=0).reset_index(drop=True)

def predict_one(row):
    try:
        aspects = get_aspects(row['review_text'])
        target = row['aspect'].lower().strip()

        for a in aspects:
            if a.get('aspect', '').lower().strip() == target:
                sentiment = a.get('sentiment', '').lower().strip()
                if sentiment in ['positive', 'neutral', 'negative']:
                    return (label_to_id[sentiment], label_to_id[row['sentiment']])
        return None
    except:
        return None


predictions = []
truths = []
failed = 0

start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(predict_one, row) for _, row in llama_sample.iterrows()]

    for i, future in enumerate(concurrent.futures.as_completed(futures)):
        result = future.result()
        if result is None:
            failed += 1
        else:
            predictions.append(result[0])
            truths.append(result[1])

        if (i + 1) % 50 == 0:
            print(f"Progress: {i + 1}/{len(llama_sample)}")

elapsed = time.time() - start
timing_log['llama_direct_baseline'] = elapsed

acc = accuracy_score(truths, predictions)
prec, rec, f1, _ = precision_recall_fscore_support(truths, predictions, average='macro', zero_division=0)

print(f"\nDirect Llama 3.1 8B baseline:")
print(f"  Samples: {len(predictions)}, failed: {failed}")
print(f"  Accuracy: {acc:.4f}, F1: {f1:.4f}")
print(f"  Time: {elapsed:.1f}s")

## Step 10: Label agreement rate

Run Llama on the held-out validation reviews, then compare against the
hand-corrected version to measure how good the silver labels are.

In [ ]:
val_df_manual = pd.read_csv(f"{base_path}\\manual_validation_set.csv")

llama_aspects_col = []
llama_sentiments_col = []

start = time.time()

for i, row in val_df_manual.iterrows():
    try:
        aspects = get_aspects(row['text'])
        asp = ", ".join([a.get('aspect', '') for a in aspects])
        sent = ", ".join([a.get('sentiment', '') for a in aspects])
    except:
        asp = ""
        sent = ""

    llama_aspects_col.append(asp)
    llama_sentiments_col.append(sent)

    if (i + 1) % 25 == 0:
        print(f"  Processed {i + 1}/{len(val_df_manual)}")

timing_log['validation_llama_labeling'] = time.time() - start

val_df_manual['my_aspects'] = llama_aspects_col
val_df_manual['my_sentiments'] = llama_sentiments_col
val_df_manual['llama_aspects_original'] = llama_aspects_col
val_df_manual['llama_sentiments_original'] = llama_sentiments_col

val_df_manual.to_csv(f"{base_path}\\manual_validation_set.csv", index=False)
print(f"\nValidation set prefilled in {timing_log['validation_llama_labeling']:.1f}s")

In [ ]:
val_df_check = pd.read_csv(f"{base_path}\\manual_validation_set.csv")

results = []
for category_name in val_df_check['category'].unique():
    rows = val_df_check[val_df_check['category'] == category_name]

    matches = 0
    total = 0

    for _, row in rows.iterrows():
        manual = str(row['my_sentiments']).split(',')
        llama = str(row['llama_sentiments_original']).split(',')

        for m, l in zip(manual, llama):
            m = m.strip().lower()
            l = l.strip().lower()
            if m or l:
                total += 1
                if m == l and m:
                    matches += 1

    results.append({
        'category': category_name,
        'reviews': len(rows),
        'matches': matches,
        'total': total,
        'agreement': matches / total if total else 0
    })

agreement_df = pd.DataFrame(results)
agreement_df

## Step 11: Visualizations

In [ ]:
modernbert_for_plot = modernbert_df[['category', 'accuracy']].copy()
modernbert_for_plot['model'] = 'ModernBERT'

combined_results = pd.concat([modernbert_for_plot, baseline_df_results[['category', 'model', 'accuracy']]])
accuracy_pivot = combined_results.pivot(index='category', columns='model', values='accuracy')

accuracy_pivot.plot(kind='bar', figsize=(12, 8))
plt.title('Model Accuracy by Category')
plt.ylabel('Accuracy')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{base_path}\\accuracy_by_category.png")
plt.show()

### Timing summary

In [ ]:
import pandas as pd

timing_summary = pd.DataFrame([
    {'step': 'Silver labels generation (Llama 8 3b)', 'seconds': 5672.2},
    {'step': 'ModernBERT fine-tuning', 'seconds': 622.2},
    {'step': 'Baselines training total', 'seconds': 123.6},
    {'step': 'Direct Llama 3 baseline', 'seconds': 572.1},
    {'step': 'Validation Llama labeling', 'seconds': 293.3},
])

timing_summary['minutes'] = (timing_summary['seconds'] / 60).round(1)
timing_summary.to_csv(f"{base_path}\\timing_summary.csv", index=False)
timing_summary

In [ ]:
all_silver = []
for category_name in categories:
    all_silver.append(pd.read_csv(f"{base_path}\\{category_name}_silver_labels.csv"))
silver_labels_df = pd.concat(all_silver, ignore_index=True)

valid = silver_labels_df[silver_labels_df['sentiment'].isin(['positive', 'neutral', 'negative'])]
sentiment_counts = valid.groupby(['category', 'sentiment']).size().unstack(fill_value=0)
sentiment_counts.plot(kind='bar', figsize=(12, 6))
plt.title('Sentiment Distribution by Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{base_path}\\sentiment_distribution.png")
plt.show()

### Per-category ROC-AUC

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
from sklearn.metrics import roc_auc_score

# load model
model_path = f"{base_path}\\modernbert_finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
trainer = Trainer(model=model)

# get ModernBERT ROC-AUC per category
modernbert_auc_results = []

for category_name in ["Electronics", "Beauty_and_Personal_Care", "Video_Games", "Amazon_Fashion", "Books"]:
    category_test = test_df[test_df['category'] == category_name]

    test_ds = Dataset.from_pandas(category_test[['input_text', 'sentiment']].reset_index(drop=True))
    test_ds = test_ds.map(lambda x: {'labels': label_to_id[x['sentiment']]})
    test_ds = test_ds.map(lambda b: tokenizer(b['input_text'], padding='max_length', truncation=True, max_length=256), batched=True)

    preds = trainer.predict(test_ds)
    probabilities = torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()

    auc = roc_auc_score(preds.label_ids, probabilities, multi_class='ovr', average='macro')
    modernbert_auc_results.append({'category': category_name, 'model': 'ModernBERT', 'roc_auc': auc})
    print(f"{category_name}: {auc:.4f}")

modernbert_auc_df = pd.DataFrame(modernbert_auc_results)

# combined chart
combined_auc = pd.concat([
    baseline_df_results[['category', 'model', 'roc_auc']],
    modernbert_auc_df
])

auc_pivot = combined_auc.pivot(index='category', columns='model', values='roc_auc')
auc_pivot.plot(kind='bar', figsize=(12, 6))
plt.title('ROC-AUC by Category')
plt.ylabel('ROC-AUC')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{base_path}\\roc_auc_by_category.png")
plt.show()

### Temporal trend data

In [ ]:
all_silver = []
for category_name in ["Electronics", "Beauty_and_Personal_Care", "Video_Games", "Amazon_Fashion", "Books"]:
    all_silver.append(pd.read_csv(f"{base_path}\\{category_name}_silver_labels.csv"))
silver_df = pd.concat(all_silver, ignore_index=True)

valid = silver_df[silver_df['sentiment'].isin(['positive', 'neutral', 'negative'])]

silver_summary = valid.groupby('category')['sentiment'].value_counts().unstack(fill_value=0)
silver_summary['total'] = silver_summary.sum(axis=1)
silver_summary

## Checkpoint: reload saved model and data

Lets the analysis below run without re-training from scratch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from datasets import Dataset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support

base_path = r"C:\Users\sapds\Capstone"

label_to_id = {"negative": 0, "neutral": 1, "positive": 2}
id_to_label = {0: "negative", 1: "neutral", 2: "positive"}

train_df = pd.read_csv(f"{base_path}\\train.csv")
test_df = pd.read_csv(f"{base_path}\\test.csv")

tokenizer = AutoTokenizer.from_pretrained(f"{base_path}\\modernbert_finetuned")
model = AutoModelForSequenceClassification.from_pretrained(f"{base_path}\\modernbert_finetuned")
trainer = Trainer(model=model)

## Step 12: Confusion matrices

In [ ]:
test_ds = Dataset.from_pandas(test_df[['input_text', 'sentiment']].reset_index(drop=True))
test_ds = test_ds.map(lambda x: {'labels': label_to_id[x['sentiment']]})
test_ds = test_ds.map(lambda b: tokenizer(b['input_text'], padding='max_length', truncation=True, max_length=256), batched=True)

preds = trainer.predict(test_ds)
pred_labels = np.argmax(preds.predictions, axis=1)

cm = confusion_matrix(preds.label_ids, pred_labels)
display = ConfusionMatrixDisplay(cm, display_labels=['negative', 'neutral', 'positive'])

fig, ax = plt.subplots(figsize=(8, 6))
display.plot(ax=ax)
plt.title('ModernBERT Confusion Matrix')
plt.tight_layout()
plt.savefig(f"{base_path}\\confusion_matrix_modernbert.png")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['input_text'])
X_test = vectorizer.transform(test_df['input_text'])
y_train = train_df['sentiment'].map(label_to_id)
y_test = test_df['sentiment'].map(label_to_id)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": CalibratedClassifierCV(LinearSVC(max_iter=2000)),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, m) in enumerate(models.items()):
    m.fit(X_train, y_train)
    predictions = m.predict(X_test)
    cm = confusion_matrix(y_test, predictions)
    display = ConfusionMatrixDisplay(cm, display_labels=['negative', 'neutral', 'positive'])
    display.plot(ax=axes[idx])
    axes[idx].set_title(name)

plt.tight_layout()
plt.savefig(f"{base_path}\\confusion_matrices_baselines.png")
plt.show()

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', CalibratedClassifierCV(LinearSVC(max_iter=2000))),
        ('rf', RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000)
)

stacking.fit(X_train, y_train)
stacking_preds = stacking.predict(X_test)

cm = confusion_matrix(y_test, stacking_preds)
display = ConfusionMatrixDisplay(cm, display_labels=['negative', 'neutral', 'positive'])

fig, ax = plt.subplots(figsize=(8, 6))
display.plot(ax=ax)
plt.title('Stacking Ensemble Confusion Matrix')
plt.tight_layout()
plt.savefig(f"{base_path}\\confusion_matrix_stacking.png")
plt.show()

### Sentiment trend over time

In [ ]:
all_silver = []
for category_name in ["Electronics", "Beauty_and_Personal_Care", "Video_Games", "Amazon_Fashion", "Books"]:
    all_silver.append(pd.read_csv(f"{base_path}\\{category_name}_silver_labels.csv"))
silver_df = pd.concat(all_silver, ignore_index=True)
silver_df['timestamp'] = pd.to_numeric(silver_df['timestamp'], errors='coerce')
silver_df['year'] = pd.to_datetime(silver_df['timestamp'], unit='ms', errors='coerce').dt.year

valid = silver_df[silver_df['sentiment'].isin(['positive', 'neutral', 'negative'])]
valid = valid.dropna(subset=['year'])
valid = valid[valid['year'] >= 2010]

trend = valid.groupby(['category', 'year', 'sentiment']).size().unstack(fill_value=0)
trend['total'] = trend.sum(axis=1)
trend['positive_ratio'] = trend['positive'] / trend['total']
trend = trend.reset_index()

for category_name in trend['category'].unique():
    category_data = trend[trend['category'] == category_name]
    plt.plot(category_data['year'], category_data['positive_ratio'], marker='o', label=category_name)

plt.title('Positive Sentiment Ratio Over Time (2010-2023)')
plt.xlabel('Year')
plt.ylabel('Positive Ratio')
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(f"{base_path}\\sentiment_trend.png")
plt.show()

## Step 13: Small sample robustness test

Clear GPU memory, then train a second model on only 5,000 examples to see
how much the full training set actually matters.

In [ ]:
import torch
import gc

try:
    del model
except NameError:
    pass
try:
    del trainer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared")

In [ ]:
import time
from transformers import TrainingArguments, Trainer

subset = train_df.sample(5000, random_state=42)

subset_ds = Dataset.from_pandas(subset[['input_text', 'sentiment']].reset_index(drop=True))
subset_ds = subset_ds.map(lambda x: {'labels': label_to_id[x['sentiment']]})
subset_ds = subset_ds.map(lambda b: tokenizer(b['input_text'], padding='max_length', truncation=True, max_length=256), batched=True)

small_model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=3)

args = TrainingArguments(
    output_dir=f"{base_path}\\modernbert_5000",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    fp16=True,
    report_to="none",
    save_strategy="no"
)

small_trainer = Trainer(model=small_model, args=args, train_dataset=subset_ds)

start = time.time()
small_trainer.train()
train_time = time.time() - start

preds = small_trainer.predict(test_ds)
pred_labels = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(preds.label_ids, pred_labels)
_, _, f1, _ = precision_recall_fscore_support(preds.label_ids, pred_labels, average='macro', zero_division=0)

print(f"5000 samples: acc={acc:.4f}, f1={f1:.4f}")
print(f"Full 29,793 samples: acc=0.767, f1=0.693")

## Step 14: Aspect rankings per category

Score each aspect as (share positive) minus (share negative), so +1 means every
mention was positive and -1 means every mention was negative. Only aspects
mentioned at least 20 times are included, otherwise one-off aspects dominate
the top and bottom of the list.

In [ ]:
all_silver = []
for category_name in categories:
    all_silver.append(pd.read_csv(f"{base_path}\\{category_name}_silver_labels.csv"))
silver = pd.concat(all_silver, ignore_index=True)

silver = silver[silver['sentiment'].isin(['positive', 'neutral', 'negative'])]
silver['aspect'] = silver['aspect'].str.lower().str.strip()

rows = []
for category_name in silver['category'].unique():
    sub = silver[silver['category'] == category_name]

    counts = sub.groupby('aspect').size()
    common = counts[counts >= 20].index
    sub = sub[sub['aspect'].isin(common)]

    score = sub.groupby('aspect')['sentiment'].apply(
        lambda x: (x == 'positive').mean() - (x == 'negative').mean()
    ).sort_values()

    for aspect in score.tail(3).index[::-1]:
        rows.append({'category': category_name, 'type': 'most positive',
                     'aspect': aspect, 'score': round(score[aspect], 2)})
    for aspect in score.head(3).index:
        rows.append({'category': category_name, 'type': 'most negative',
                     'aspect': aspect, 'score': round(score[aspect], 2)})

aspect_rankings = pd.DataFrame(rows)
aspect_rankings.to_csv(f"{base_path}\\aspect_rankings.csv", index=False)
aspect_rankings

## Appendix: reported figures

Reproduces the summary numbers cited in the report and slides.

In [ ]:
import pandas as pd

# baseline metrics, averaged across the five categories
baselines = pd.read_csv(f"{base_path}\\baseline_per_category.csv")
print("Baseline averages across categories:")
print(baselines.groupby('model')[['accuracy', 'f1', 'precision', 'recall', 'roc_auc']].mean().round(3))

# ModernBERT inference throughput
modernbert = pd.read_csv(f"{base_path}\\modernbert_per_category.csv")
print(f"\nModernBERT throughput: {modernbert['reviews_per_second'].mean():.1f} reviews/sec")

# silver label class distribution
all_silver = []
for category_name in categories:
    all_silver.append(pd.read_csv(f"{base_path}\\{category_name}_silver_labels.csv"))
silver = pd.concat(all_silver, ignore_index=True)
silver = silver[silver['sentiment'].isin(['positive', 'neutral', 'negative'])]

summary = silver.groupby('category')['sentiment'].value_counts().unstack(fill_value=0)
summary['total'] = summary.sum(axis=1)
print("\nSilver label distribution:")
print(summary)